In [0]:
dbutils.widgets.text("IOTconfigId", "24")
IOTconfigId = dbutils.widgets.get("IOTconfigId")

dbutils.widgets.text("IOTsourceTypeId", "8")
IOTsourceTypeId = dbutils.widgets.get("IOTsourceTypeId")

dbutils.widgets.text("USBconfigId", "31")
USBconfigId = dbutils.widgets.get("USBconfigId")

dbutils.widgets.text("USBsourceTypeId", "9")
USBsourceTypeId = dbutils.widgets.get("USBsourceTypeId")

dbutils.widgets.text("destinationContainerPath", "raw")
destinationContainerPath = dbutils.widgets.get("destinationContainerPath")

dbutils.widgets.text("destinationFolderPath", "DeviceLogs/COM1/<DeviceId>/<LoadType>/<YYYYMMDD>/<LogType>")
destinationFolderPath = dbutils.widgets.get("destinationFolderPath")

dbutils.widgets.text("parent_run_id", "1")
pipeLineId = dbutils.widgets.get("parent_run_id")

dbutils.widgets.text("job_id", "1")
jobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

In [0]:

%run ../Configurations/Init_Scripts

In [0]:
import json
import multiprocessing
import os
from multiprocessing.pool import ThreadPool,Pool
from azure.servicebus import ServiceBusClient, ServiceBusMessage, AutoLockRenewer, NEXT_AVAILABLE_SESSION
from azure.servicebus.exceptions import OperationTimeoutError
from azure.servicebus.management import MessageCountDetails
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceNotFoundError
from datetime import datetime
import traceback
import time

In [0]:
def MessageProcessing(message,receiver, blobStorageConnectionString):
    filePrefix = '/dbfs/mnt'
    loadType = ''
    deviceDate = ''
    deviceId = ''
    logType = ''
    destinationFolderPathDerived = ''
    destinationFileName = ''
    processedFileList = []

    fileDetails = dict()
    fileDetails['DeviceType'] = 'COM1'
    # fileDetails['ConfigId'] = configId
    # fileDetails['SourceTypeId'] = sourceTypeId
    fileDetails['PipelineRunId'] = pipeLineId
    fileDetails['JobId'] = jobId    
    
    try:

        message_json = json.loads(str(message))
        eventType = message_json.get('eventType')
        subject = message_json.get('subject')
        extension = subject.split('.')[-1]
        sourceFilePath = subject.replace('/blobServices/default/containers','').replace('/blobs','')
        sourceFilePathList = sourceFilePath.split('/')
        sourceContainerPath = sourceFilePathList[1]
        sourceFolderPath = sourceFilePath[sourceFilePath.find('/',1)+1:sourceFilePath.rfind('/',1)]
        sourceFileName = sourceFilePathList[-1]
        logList = {'EX' : 'Exception','xGate1' : 'AgentLog', 'UE' : 'UserException', 'UL' : 'Usage', 'Assert' : 'Assert', 'xGate' : 'AgentLog',
                  'DeployStatus' : 'DeployStatus', 'Unparsed' : 'Other', 'ZInstaller' : 'InstallLogs' , 'LastContactReport' : 'COM1LastConnected',
                   'COM1' : 'Measurement'}

        if('usb' in sourceFilePath.lower().split('/')):
            loadType = 'USB'
            fileDetails['ConfigId'] = USBconfigId
            fileDetails['SourceTypeId'] = USBsourceTypeId
        else:
            loadType = 'IOT'
            fileDetails['ConfigId'] = IOTconfigId
            fileDetails['SourceTypeId'] = IOTsourceTypeId
        
        deviceDate = sourceFileName.replace("_","")[-18:-10]
        if not(deviceDate.isnumeric()) or (len(sourceFileName) < 20):
            deviceDate = datetime.utcnow().strftime("%Y%m%d")
        if(len(sourceFilePathList)>=2) and (len(sourceFileName) > 20):
            for logListKey in logList:
                if(logListKey in sourceFileName):
                    logType = logList[logListKey]
                    break
            if(logType == ''):
                logType = 'InValid'
                    
            #logType = sourceFilePathList[2]
            if(logType == 'DeployStatus'):
                #09-30-22.12-00-30-DeployStatus
                year = '20'+sourceFileName[6:8]
                month = sourceFileName[0:2]
                day = sourceFileName[3:5]
                deviceDate = year+month+day #creating date format for folderpath
            elif(logType == 'Other'):
                deviceId = sourceFileName.split('_')[2].replace(" ","")
                deviceId = str(deviceId[:16]).upper()
            elif(logType == 'InstallLogs'):
                deviceId = sourceFileName.split('_')[1].replace("-","")
                deviceId = str(deviceId[:16]).upper()
            elif(sourceFileName.lower().find('unparsed_')>0):
                deviceId = sourceFileName.split('_')[2].replace(" ","")
                deviceId = str(deviceId[:16]).upper()
            elif(len(sourceFileName.split('_'))>=2):
                deviceId = sourceFileName.split('_')[1].replace(" ","")
                deviceId = str(deviceId[:16]).upper()
            else:
                deviceId = 'InValid'
        else:
            deviceId = 'InValid'
            
            
        sourcePath = filePrefix+'/'+sourceContainerPath+'/'+sourceFolderPath+'/'+sourceFileName
        destinationFileName  = sourceFileName

        if(deviceId == 'InValid'):
            destinationFolderPathDerived = destinationFolderPath.replace('<DeviceId>','').replace('<LoadType>',loadType).replace('<LogType>','').replace('<YYYYMMDD>','')
        else:   
            destinationFolderPathDerived = destinationFolderPath.replace('<DeviceId>',deviceId).replace('<LoadType>',loadType).replace('<LogType>',logType).replace('<YYYYMMDD>',deviceDate)
            
            
        destinationPath = filePrefix+'/'+destinationContainerPath+'/'+destinationFolderPathDerived+'/'+destinationFileName
        
    except Exception as exp:
        print(message)
        print(str(exp))
        fileDetails['Status'] = 'Failed'
        fileDetails['ErrorMessage'] = str(exp)
        traceback.print_exc()
#         raise
        return fileDetails        

    fileDetails['SourceFolderPath'] = '/'+sourceFolderPath+'/'
    fileDetails['SourceFileName'] = sourceFileName
    fileDetails['SourceContainerPath'] = sourceContainerPath
    fileDetails['DestinationFolderPath'] = '/'+destinationFolderPathDerived+'/'
    fileDetails['DestinationFileName'] = destinationFileName
    fileDetails['DestinationContainerPath'] = destinationContainerPath
    fileDetails['DeviceId'] = deviceId
    fileDetails['LoadType'] = loadType
    fileDetails['LogType'] = logType   
    try: 
        # Get Parent file name
        if(extension.lower()!='zip'):        
            container_name="logs-com1-raw"        
            blob_name=sourceFolderPath+'/'+sourceFileName;
            with BlobServiceClient.from_connection_string(blobStorageConnectionString) as blob_service_client:
                blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
                blob_properties = blob_client.get_blob_properties()
                metadata = {key.lower(): val for key, val in blob_properties.metadata.items()}                           
                if "parentfilename" in metadata:
                    fileDetails['ZipFileName']=metadata['parentfilename']          
    except  ResourceNotFoundError:
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = 'BlobNotFound' 
        try:     
            (receiver.dead_letter_message(message,
                                        reason=fileDetails['PipelineStatus'],
                                        error_description=fileDetails['ErrorMessage']))
        except:
            pass
        return fileDetails['ErrorMessage'],fileDetails
    except Exception as e:
        print("Error:", e)
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = str(e)
        return fileDetails['ErrorMessage'],fileDetails

    try:    
        if(eventType == "Microsoft.Storage.BlobCreated" and extension.lower() in ('csv','txt','log') and sourceContainerPath.lower() == 'logs-com1-raw'):
            
            
            retry_value=3
            while retry_value>0:
                try:
                    copyFiles(sourcePath,destinationPath)
                    # dbutils.fs.cp(sourcePath.replace("/dbfs",""),destinationPath.replace("/dbfs",""))
                    retry_value=0
                except Exception as e:
                    retry_value=retry_value-1
                    if retry_value<=0:
                        raise
                    time.sleep(10)
            
            fileDetails['SourceFileSize'] = os.path.getsize(sourcePath)            
            if(deviceId == 'InValid'):
                if str(sourceFileName).lower().find('lastcontactreport')>0:
                    fileDetails['PipelineStatus'] = 'NotApplicable'
                    fileDetails['ErrorMessage'] = ''
                    receiver.complete_message(message)
                    return '',fileDetails
                else:
                    fileDetails['PipelineStatus'] = 'NotApplicable'
                    fileDetails['ErrorMessage'] = 'Invalid log type and invalid file name'
                    (receiver.dead_letter_message(message,
                                                reason=fileDetails['PipelineStatus'],
                                                error_description=fileDetails['ErrorMessage']))
                    return fileDetails['ErrorMessage'],fileDetails
            else:
                fileDetails['PipelineStatus'] = 'Succeeded'
                fileDetails['ErrorMessage'] = ''
                # processedFileList.append(fileDetails)                
                # logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')
                receiver.complete_message(message)                
                return '',fileDetails
        else:
            fileDetails['PipelineStatus'] = 'NotApplicable'
            fileDetails['ErrorMessage'] = 'inValid File Extension not in csv,txt,log or not in logs-com1-raw container or event type is not blob created'
            # processedFileList.append(fileDetails)            
            # logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')            
            receiver.complete_message(message)
            return ''  ,fileDetails
    except Exception as exp:
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = str(exp)
        # processedFileList.append(fileDetails)        
        # logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')   
        try:     
            (receiver.dead_letter_message(message,
                                        reason=fileDetails['PipelineStatus'],
                                        error_description=fileDetails['ErrorMessage']))
        except:
            pass
        # traceback.print_exc()
#         raise
        return fileDetails['ErrorMessage'],fileDetails


In [0]:
def ReceiveMessageIterative(connection_string, queue_name, blobStorageConnectionString):
    try:
        cpuCount = os.cpu_count()
        executors=30*cpuCount
        message_count=30*cpuCount
        exceptionList = []
        messageProcessed=0
        renewer = AutoLockRenewer() 
        pool = ThreadPool(executors)       
        with ServiceBusClient.from_connection_string(connection_string) as sb_client:
            with sb_client.get_queue_receiver(queue_name,prefetch_count=message_count) as receiver:
                while True:
                    # renewer = AutoLockRenewer() 
                    currentTime=datetime.utcnow()
                    receivedMessages = receiver.receive_messages(max_message_count=message_count,max_wait_time=5)
                    print("receivedMessages:"+str(len(receivedMessages)))
                    if(len(receivedMessages)>0):
                        processedFileList = []

                        exceptionList = []
                        allMessages = []
                        for message in receivedMessages:
                            renewer.register(receiver, message, max_lock_renewal_duration=6000)
                            allMessages = allMessages + [(message, receiver, blobStorageConnectionString)]

                        print("allMessages:"+str(len(allMessages)))
                        dataDictList=[]
                        strList=[]
                        # pool = ThreadPool(executors)
                         
                        returnList= pool.starmap(MessageProcessing, allMessages)
                        
                        for returndata in returnList:
                            strList.append(returndata[0])
                            dataDictList.append(returndata[1])
                        print("strList:"+str(len(strList)))
                        print("dataDictList:"+str(len(dataDictList)))
                       
                        logIntoIngestionLogTable(dataDictList,'LoadLogFiles_COM1') 
                        # pool.close()
                        # pool.join()
                        
                        for returnValue in strList:
                            if returnValue:
                                exceptionList.append(returnValue)
                        messageProcessed=messageProcessed+len(receivedMessages)
                        processTime=datetime.utcnow()-currentTime
                        print("Time taken to process "+str(len(receivedMessages))+" is "+str(processTime.seconds/60)+" minute.")
                        print("--------------------------------------------------------------------------------------------------------------------------------------------")
                         
                    else:
                        print("TotalMessageProcessed:"+str(messageProcessed))
                        if(exceptionList):
                            print(exceptionList)
                            dbutils.notebook.exit((exceptionList,messageProcessed))
                            raise Exception('Exception raised while processing files and details above')

                        break
                print('Done..!')
                pool.close()
                pool.join()
                renewer.close() 
                dbutils.notebook.exit(("Succeeded",messageProcessed))
                spark.sql("clear cache")
    except OperationTimeoutError:
        print("If timeout occurs during connecting to a session,It indicates that there might be no non-empty sessions remaining; exiting.This may present as a UserError in the azure portal metric.")
        raise

In [0]:
serviceBusConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="CSServiceBus")
com1IngestionQueueName = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="ABV-Com1IngestionQueueName")
blobStorageConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="BlobStorageConnectionString")

ReceiveMessageIterative(serviceBusConnectionString, com1IngestionQueueName, blobStorageConnectionString)